In [79]:
# 相关包的导入
import numpy as np
import pandas as pd

from datetime import datetime

import os

from climate_indices import indices
from climate_indices import compute  # 计算SPEI的包

workpath=r'G:\drought'
data_path=os.path.join(workpath,'data','SPEI_climate_indices')
daily_data_path=os.path.join(workpath,'data','dailyclimate缺测插补')

SPEI_data_path=os.path.join(data_path,'month')
SPEI_Merge_Path=os.path.join(data_path,'csv')
SPEI_Shp_Path=os.path.join(data_path,'month_shp')
stinfo=pd.read_excel(os.path.join(data_path,'站点编号及信息_选定_1969-2022.xlsx'))
stinfo['station'] = stinfo['station'].apply(str)
station_list=list(stinfo['station'])
lat_list=list(stinfo['lat'])
lon_list=list(stinfo['lon'])  
st_sum=len(station_list)

dataname='clmSPEI'

In [80]:
startyear,endyear=1969,2022
for i in range(st_sum): 
    st_no=stinfo['station'][i]
    lat,lon=stinfo['lat'][i],stinfo['lon'][i]
    
    df=pd.read_csv(os.path.join(daily_data_path,st_no+'.csv'))    
    df['time']=pd.to_datetime(df['time'])
    df=df[df['time']>=datetime(1969,1,1)]
    df['year']=df['time'].dt.year
    df['month']=df['time'].dt.month
    df=df.groupby(['year','month']).agg({'R20': [np.sum], 'T': [np.mean]}).reset_index()
    df.columns=['year','month','R20_sum','T_mean']

    tas_data=np.asarray(df['T_mean'])
    pre_data=np.asarray(df['R20_sum'])
    
    outdf=pd.DataFrame()    
    outdf['year']=df['year']
    outdf['month']=df['month']
    
    # 潜在蒸散发计算-桑斯维特方法
    pet_data = indices.pet(temperature_celsius=tas_data,latitude_degrees=lat,data_start_year=startyear)
    t_list=list(range(1,13))
    t_list.append(24)

    for t in t_list:
        # 计算SPEI
        spei = indices.spei(precips_mm=pre_data,
                        pet_mm=pet_data, # pet_mm=tas_data是不正确的,
                        scale=t, # 累积时间尺度t个月
                        distribution=indices.Distribution.gamma,
                        periodicity=compute.Periodicity.monthly,
                        data_start_year=endyear,
                        calibration_year_initial=startyear,
                        calibration_year_final=endyear,
                        )
        outdf[dataname+str(t)+'_Month']=spei
    outdf.to_csv(os.path.join(SPEI_data_path,st_no+'.csv'))